##### **Objective:** Prepare the dataset for machine learning in a reproducible, leakage-free manner.
- All preprocessing decisions will be based on observations from previous phases (Data Understanding and EDA).

In [1]:
import pandas as pd 
from sklearn.model_selection import train_test_split


In [2]:
# Load the raw train.csv and test.csv datasets
train_df = pd.read_csv('../data/raw/train.csv')
test_df = pd.read_csv('../data/raw/test.csv')

# Separate target and predictors and remove the SalePrice column from the feature matrix. 
y =train_df['SalePrice']
X = train_df.drop(columns="SalePrice") # Ensures readability


# Perform a train/validation split on the training data.
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=42, shuffle=True, stratify=None)

# 42 is a reference to Douglas Adams's The Hitchhiker's Guide to the Galaxy.

# shuffle = True ensures that the training and validation sets are representative of the overall dataset. It randomly picks 20% of rows from anywhere in the dataset (because shuffle=True is the default setting behind the scenes).


#Inspect the shapes
print(f"Training features: {X_train.shape}")
print(f"Validation features: {X_valid.shape}")
print(f"Training target: {y_train.shape}")
print(f"Validation target: {y_valid.shape}")



Training features: (1168, 80)
Validation features: (292, 80)
Training target: (1168,)
Validation target: (292,)


- Stratify must be kept none because stratification is designed for categorical classification targets (like predicting 1 or 0) to ensure each fold gets equal class proportions. But SalePrice is a continuous numeric variable (regression target). So it gies an error like: The least populated classes in y have only 1 member, which is too few. The minimum number of groups for any class cannot be less than 2. Classes with too few members are: [34900, 35311, 37900, ...]

- train_test_split attempts to preserve class proportions. Since every SalePrice is effectively unique, each value becomes its own class, making stratification impossible.

- `train_test_split(X, y, test_size=0.2, shuffle=False)`: Takes the bottom 20% of rows sequentially without shuffling.

In [3]:
#Veriffy split:
print("First few rows of X_train:")
X_train.head()

First few rows of X_train:


,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,ScreenPorch,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition
254,255,20,RL,70.0,8400,Pave,NaN,Reg,Lvl,AllPub,...,0,0,NaN,NaN,NaN,0,6,2010,WD,Normal
1066,1067,60,RL,59.0,7837,Pave,NaN,IR1,Lvl,AllPub,...,0,0,NaN,NaN,NaN,0,5,2009,WD,Normal
638,639,30,RL,67.0,8777,Pave,NaN,Reg,Lvl,AllPub,...,0,0,NaN,MnPrv,NaN,0,5,2008,WD,Normal
799,800,50,RL,60.0,7200,Pave,NaN,Reg,Lvl,AllPub,...,0,0,NaN,MnPrv,NaN,0,6,2007,WD,Normal
380,381,50,RL,50.0,5000,Pave,Pave,Reg,Lvl,AllPub,...,0,0,NaN,NaN,NaN,0,5,2010,WD,Normal


In [4]:
print("First few rows of X_valid:")
X_valid.head()

First few rows of X_valid:


,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,ScreenPorch,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition
892,893,20,RL,70.0,8414,Pave,NaN,Reg,Lvl,AllPub,...,0,0,NaN,MnPrv,NaN,0,2,2006,WD,Normal
1105,1106,60,RL,98.0,12256,Pave,NaN,IR1,Lvl,AllPub,...,0,0,NaN,NaN,NaN,0,4,2010,WD,Normal
413,414,30,RM,56.0,8960,Pave,Grvl,Reg,Lvl,AllPub,...,0,0,NaN,NaN,NaN,0,3,2010,WD,Normal
522,523,50,RM,50.0,5000,Pave,NaN,Reg,Lvl,AllPub,...,0,0,NaN,NaN,NaN,0,10,2006,WD,Normal
1036,1037,20,RL,89.0,12898,Pave,NaN,IR1,HLS,AllPub,...,0,0,NaN,NaN,NaN,0,9,2009,WD,Normal


# **Investigating and Handling Null Values**

## Deep Investigation into Suspecious Features:

In [18]:
train_df.loc[train_df["MasVnrArea"].isnull(), ["Id", "MasVnrType", "MasVnrArea"]]


,Id,MasVnrType,MasVnrArea
234,235,NaN,NaN
529,530,NaN,NaN
650,651,NaN,NaN
936,937,NaN,NaN
973,974,NaN,NaN
977,978,NaN,NaN
1243,1244,NaN,NaN
1278,1279,NaN,NaN


In [19]:
train_df.loc[(train_df["MasVnrArea"].notnull() & train_df["MasVnrType"].isnull()), ["Id", "MasVnrType", "MasVnrArea"]]

,Id,MasVnrType,MasVnrArea
1,2,NaN,0.0
3,4,NaN,0.0
5,6,NaN,0.0
8,9,NaN,0.0
9,10,NaN,0.0
...,...,...,...
1454,1455,NaN,0.0
1455,1456,NaN,0.0
1457,1458,NaN,0.0
1458,1459,NaN,0.0


In [ ]:
train_df.loc[(train_df["MasVnrArea"] == 0) & (train_df["MasVnrType"].notnull()), ["Id", "MasVnrType", "MasVnrArea"]]

,Id,MasVnrType,MasVnrArea
688,689,BrkFace,0.0
1241,1242,Stone,0.0


In [46]:
train_df.loc[(train_df["MasVnrArea"].isnull()) & (train_df["MasVnrType"].notnull()), ["Id", "MasVnrType", "MasVnrArea"]]

,Id,MasVnrType,MasVnrArea


In [23]:
train_df.loc[(train_df["Fireplaces"] == 0) & (train_df["FireplaceQu"].notnull()), ["Id", "Fireplaces", "FireplaceQu"]]

,Id,Fireplaces,FireplaceQu


In [30]:
train_df.loc[train_df["LotFrontage"] == 0]

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice


In [36]:
train_df.loc[(train_df["BsmtExposure"].isnull()) & (train_df["BsmtCond"].notnull()), ["Id", "BsmtCond", "BsmtExposure", "BsmtFinType2", "BsmtQual", "BsmtFinType1", "BsmtFinType2"]]

,Id,BsmtCond,BsmtExposure,BsmtFinType2,BsmtQual,BsmtFinType1,BsmtFinType2
948,949,TA,NaN,Unf,Gd,Unf,Unf


In [ ]:
train_df.loc[(train_df["BsmtFinType2"].isnull()) & (train_df["BsmtCond"].notnull()), ["Id", ]]"BsmtCond", "BsmtExposure", "BsmtFinType2", "BsmtQual", "BsmtFinType1", "BsmtFinType2"

,Id,BsmtCond,BsmtExposure,BsmtFinType2,BsmtQual,BsmtFinType1,BsmtFinType2
332,333,TA,No,NaN,Gd,GLQ,NaN


#### Handling Missing Values


In [ ]:
# avoid GarageYrBlt, as if we replace missing value in it as 0, it will become outlier.
cat_structural_cols = [
    'Alley', 'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 
    'BsmtFinType2', 'FireplaceQu', 'GarageType', 'GarageFinish', 
    'GarageQual', 'GarageCond', 'PoolQC', 'Fence', 'MiscFeature'
]
# "Needs further validation against MasVnrType."
num_structural_cols = [
    'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', 
    'BsmtFullBath', 'BsmtHalfBath', 'GarageCars', 'GarageArea', 
    'MasVnrArea'
]
cat_fill_dict = {col: 'None' for col in cat_structural_cols}
num_fill_dict = {col: 0 for col in num_structural_cols} 

structural_fill_dict = {**cat_fill_dict, **num_fill_dict}

X_train = X_train.fillna(value = structural_fill_dict)
X_valid = X_valid.fillna(value = structural_fill_dict)
test_df = test_df.fillna(value = structural_fill_dict)

In [44]:
# Remaining structural missingness
train_rem = X_train[cat_structural_cols + num_structural_cols].isnull().sum().sum()
valid_rem = X_valid[cat_structural_cols + num_structural_cols].isnull().sum().sum()
test_rem  = test_df[cat_structural_cols + num_structural_cols].isnull().sum().sum()

print(f"Remaining structural NaNs in X_train: {train_rem}")
print(f"Remaining structural NaNs in X_valid: {valid_rem}")
print(f"Remaining structural NaNs in test_df: {test_rem}")

Remaining structural NaNs in X_train: 0
Remaining structural NaNs in X_valid: 0
Remaining structural NaNs in test_df: 0


> **Note**: The handling of structural missingness was done mostly by using the competition documentation so that rules (example NaN means no pool) comes from external knowledge, not from the training distribution. This means this particular transformation is deterministic (i.e. it doesn't estimate anything from the data.) Such transformations does not have a fit stage.

**Deterministic/Rule Based Transformations:**
- Rename columns
- Parse dates
- Replace structural NaNs
- Convert units
- Strip whitespace

**Transformations where Data is Learned:**
- Mean imputation
- Median imputation
- StandardScaler
- PCA
- KNN Imputer
- Target Encoding